# ANN Project — Kaggle GPU Pipeline

Full end-to-end GPU execution for the adversarial AI-text detection pipeline.

**Pipeline**: Train (4 ablations) → Evaluate → Benchmark → Decision Gate → Multi-Seed

---
### Prerequisites (do once before running this notebook)

1. **Attach preprocessed RAID dataset**
   - Upload your `data/processed/` parquet files as a Kaggle Dataset
   - Add it to this notebook via Notebook → Add Data (top-right)
   - Expected files: `raid_train_pool.parquet`, `raid_unseen_pool.parquet`, etc.

2. **Create results Dataset handle** (optional, for cross-session persistence)
   - Create an empty Kaggle Dataset named `ann-project-results` under your account
   - The upload will create/update versions automatically

---
### How to run

1. **Session 1** (first 9h): Run cells in order → G0 + G2 + partial G1
2. **Session 2** (next 9h, with resume): Run cells again → resume G1 + G3
3. **Session 3** (final 9h, with resume): Resume G1 if needed → G5

The `--resume` flag (default: on) detects completed ablations and skips them.
Results persist via Kaggle Dataset upload at the end of each session.

---
## Cell 1: Clone the repository

In [ ]:
# Option A: Clone from GitHub (requires committed + pushed code)
!git clone https://github.com/The-Immortal-Wanderer/Adversarially-Robust-AI-Text-Detection-via-Supervised-DistilBERT-Fine-Tuning.git
!mv Adversarially-Robust-AI-Text-Detection-via-Supervised-DistilBERT-Fine-Tuning/* ./
!mv Adversarially-Robust-AI-Text-Detection-via-Supervised-DistilBERT-Fine-Tuning/.* ./ 2>/dev/null || true
!rmdir Adversarially-Robust-AI-Text-Detection-via-Supervised-DistilBERT-Fine-Tuning/

# Option B: Zip upload — skip the above, instead uncomment below:
# !unzip -q /kaggle/input/your-dataset-name/ann_project.zip -d .

In [ ]:
print('Repository files:')
!ls -la

---
## Cell 2: Environment check

Verify GPU is available and print hardware info.

In [ ]:
import torch, sys, platform
print(f"Python      : {sys.version.split()[0]}")
print(f"PyTorch     : {torch.__version__}")
print(f"CUDA avail  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name}  {p.total_memory / 1e9:.1f}GB  Compute {p.major}.{p.minor}")
print(f"Platform    : {platform.platform()}")

---
## Cell 3: Full pipeline — Train, Evaluate, Benchmark (all 4 ablations)

This runs `scripts/kaggle_run.py` which:
- Installs dependencies from `requirements.txt`
- Copies preprocessed data from `/kaggle/input/` to `data/processed/`
- Trains all 4 ablations (baseline1, ablation_b, ablation_a, ablation_c) sequentially
- Evaluates each on the unseen split
- Runs benchmark.py latency tests
- Uploads results archive to Kaggle Dataset at end of session
- **Auto-resumes** on next session (skips completed ablations)

**Toggle flags**:
- `--seed 42` — single-seed for G0/G2 (change to `--seed 42 123 456 789 101112` for full G1)
- `--no-resume` — force re-run all ablations from scratch
- `--upload` / `--no-upload` — control results archive upload

In [ ]:
!python scripts/kaggle_run.py \
    --seed 42 \
    --resume \
    --run-log-path /kaggle/working/run_log.json \
    --upload

---
## Cell 4: Pre-G1 Decision Gate

Runs single-seed evaluation on all 4 ablations and computes RRD spread.
**Decision**: If RRD spread < 2%, proceed to multi-seed (G1). If >2%,
the narrative needs a fallback contingency.

In [ ]:
!python scripts/g0_decision_gate.py \
    --seed 42 \
    --checkpoint-dir artifacts/distilbert_detector \
    --results-dir results

---
## Cell 5: Multi-Seed Evaluation (G1) — 5 seeds × 4 configs = 20 runs

**Only run this AFTER G0 passes (<2% RRD spread).**

This will take ~14h. The `--resume` flag handles multi-session execution:
- Session 1: runs ~10 of 20, uploads results
- Session 2: resumes where it left off, runs remaining 10, uploads

In [ ]:
# G1 — Multi-Seed
!python scripts/kaggle_run.py \
    --seed 42 123 456 789 101112 \
    --resume \
    --run-log-path /kaggle/working/run_log.json \
    --upload

---
## Cell 6: Clean-Only Baseline (G2) — ~6h

Train on human-genuine texts only (no AI-generated samples in training).
Can run in parallel with G0 if dual T4 available.

In [ ]:
# G2 — Clean-Only Baseline
!python scripts/train.py \
    --config ablations ablation_b \
    --training.seed 42 \
    --training.ablation_name clean_baseline \
    --training.num_epochs 20 \
    --upload

---
## Cell 7: Zero-Shot Baselines on Dedup'd Data (G3) — ~4h

Run PerplexityBaseline and Binoculars on the dedup'd unseen split.

In [ ]:
# G3 — Perplexity Baseline (dedup'd)
!python scripts/evaluate.py \
    --baseline perplexity \
    --seed 42 \
    --output results/baseline_perplexity_dedup.json

In [ ]:
# G3 — Binoculars Baseline (dedup'd)
!python scripts/evaluate.py \
    --baseline binoculars \
    --seed 42 \
    --output results/baseline_binoculars_dedup.json

---
## Cell 8: Calibration/ECE + Brier Score (G5) — ~3h

Post-G1 only. Requires G1 checkpoints with saved probability arrays.

In [ ]:
# G5 — Calibration analysis
!python scripts/evaluate.py \
    --checkpoint artifacts/distilbert_detector \
    --compute-calibration \
    --output results/calibration_metrics.json

---
## Cell 9: Upload results archive manually (if auto-upload failed)

The previous cells auto-upload on completion. If something went wrong,
use this cell to manually create and upload the results archive.

In [ ]:
# Manual upload fallback
import tarfile, tempfile, shutil
import kagglehub

archive_path = Path('/kaggle/working/results_archive.tar.gz')
with tarfile.open(archive_path, 'w:gz') as tar:
    for d in ['artifacts', 'results', 'benchmark_logs']:
        p = Path('/kaggle/working') / d
        if p.exists():
            tar.add(p, arcname=d)

temp_dir = Path(tempfile.mkdtemp(prefix='ann_upload_'))
with tarfile.open(archive_path, 'r:gz') as tar:
    tar.extractall(path=temp_dir)

kagglehub.dataset_upload(
    handle='The-Immortal-Wanderer/ann-project-results',
    local_dataset_dir=str(temp_dir),
)
shutil.rmtree(temp_dir)
print('Upload complete.')